# 25-тарау. Астананың ауа райын болжаймыз

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 25.1. Жобаның мақсаты

## 25.2. 1-қадам. Деректермен танысу

In [ ]:
import numpy as np
import pandas as pd

URL = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/data/astana_temp.csv")
df = pd.read_csv(URL, parse_dates=["күн"])
print(df.shape)
print(df.head(3))

In [ ]:
monthly = df.groupby(df["күн"].dt.month)["температура"].mean()
print(monthly.round(1).tolist())

## 25.3. 2-қадам. Базалық болжамдар

In [ ]:
df["t-1"] = df["температура"].shift(1)       # кешегі температура
print(df[["күн", "температура", "t-1"]].head(3))

In [ ]:
from sklearn.metrics import mean_absolute_error as mae

doy = df["күн"].dt.dayofyear                  # жылдың нешінші күні
past = df["күн"].dt.year <= 2022
future = df["күн"].dt.year >= 2023
clim = df[past].groupby(doy[past])["температура"].mean()

y_true = df.loc[future, "температура"]
print(f"Тұрақтылық: {mae(y_true, df.loc[future, 't-1']):.2f} °C")
print(f"Климаттық:  {mae(y_true, doy[future].map(clim)):.2f} °C")

## 25.4. 3-қадам. Белгілерді құрастыру

In [ ]:
t = df["температура"]
for k in [2, 3, 7]:
    df[f"t-{k}"] = t.shift(k)
df["апта"] = t.shift(1).rolling(7).mean()    # соңғы 7 күннің орташасы

In [ ]:
angle = 2 * np.pi * doy / 365.25
df["sin"] = np.sin(angle)
df["cos"] = np.cos(angle)
df = df.dropna()            # бос мәндері бар жолдар өшіріледі
print(df.shape)

## 25.5. 4-қадам. Уақыт бойынша бөлу

In [ ]:
features = ["t-1", "t-2", "t-3", "t-7", "апта", "sin", "cos"]
train = df[df["күн"].dt.year <= 2022]
test = df[df["күн"].dt.year >= 2023]
X_train, y_train = train[features], train["температура"]
X_test, y_test = test[features], test["температура"]
print(len(train), len(test))

## 25.6. 5-қадам. Модельдерді оқыту

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

lin = LinearRegression().fit(X_train, y_train)
forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                               random_state=42).fit(X_train, y_train)
results = {
    "Климаттық": mae(y_test, test["күн"].dt.dayofyear.map(clim)),
    "Тұрақтылық": mae(y_test, test["t-1"]),
    "Кездейсоқ орман": mae(y_test, forest.predict(X_test)),
    "Сызықтық регрессия": mae(y_test, lin.predict(X_test)),
}
for name, err in results.items():
    print(f"{name:19} {err:.2f} °C")

In [ ]:
for name, w in zip(features, lin.coef_):
    print(f"{name:5} {w:+.3f}")

## 25.7. 6-қадам. Бірнеше күн алға болжау

In [ ]:
t = df["температура"]
for h in [1, 2, 3, 5, 7, 10, 14]:
    lags = pd.DataFrame({f"lag{k}": t.shift(h + k) for k in [0, 1, 2, 6]})
    data = pd.concat([lags, df[["sin", "cos", "температура"]]], axis=1)
    data = data.dropna()
    old = df.loc[data.index, "күн"].dt.year <= 2022
    X, y = data.drop(columns="температура"), data["температура"]
    m = LinearRegression().fit(X[old], y[old])
    print(f"{h:2d} күн алға: {mae(y[~old], m.predict(X[~old])):.2f} °C")

## 25.8. 7-қадам. Нақты деректермен жұмыс

In [ ]:
import requests

url = ("https://archive-api.open-meteo.com/v1/archive"
       "?latitude=51.17&longitude=71.45"
       "&start_date=2015-01-01&end_date=2024-12-31"
       "&daily=temperature_2m_mean&timezone=auto")
data = requests.get(url).json()["daily"]
real = pd.DataFrame({"күн": pd.to_datetime(data["time"]),
                     "температура": data["temperature_2m_mean"]})
print(real.tail())